In [2]:
import os
import sys
import logging
from pathlib import Path
from dotenv import load_dotenv
import torch
from torch import nn
from torch.utils.data import Dataset
import pandas as pd
from huggingface_hub import login
from datasets import load_dataset, Dataset as HFDataset
from typing import Tuple

logging.basicConfig(
    level=logging.INFO,
    format="%(name)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger()

torch.manual_seed(123)

# src_path = Path.cwd().parent / "src"
# if src_path.exists() and str(src_path) not in sys.path:
#     sys.path.insert(0, str(src_path))

# print(f"Added to sys.path: {src_path}")
load_dotenv()  # reads .env file from the current directory

PATH_DATA = Path.cwd().parent.parent / ".data"
PATH_GPT2_124M_WEIGHTS = PATH_DATA / "model_weights" / "gpt2" / "124M" / "parameters.pickle.gz"
# DATASET = "openchat/ultrachat-sharegpt"
DATASET="~/Downloads/alpaca_gpt4_data.json"

login(os.getenv("HF_TOKEN"))

httpx | INFO | HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
huggingface_hub._login | WARNING | Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    formatted_input = instruction_text + input_text + f"\n\n### Response:\n{entry['output']}"
    return {"texts": formatted_input}

class InstructionDataset(Dataset):

    def _format_input(self, entry):
        instruction_text = (
            f"Below is an instruction that describes a task. "
            f"Write a response that appropriately completes the request."
            f"\n\n### Instruction:\n{entry['instruction']}"
        )

        input_text = (
            f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
        )
        return instruction_text + input_text

    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:         #1
            instruction_plus_input = self._format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [4]:
source_ds = (load_dataset("json", data_files="../../.data/alpaca_gpt4_data.json")["train"])
ds = source_ds.train_test_split(test_size=0.2)
ds_train = ds["train"]
ds_test = ds["test"]
ds_val = ds_test.train_test_split(test_size=0.5)
ds_test = ds_val["train"]
ds_val = ds_val["test"]

httpx | INFO | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"


In [5]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
ds_train = InstructionDataset(tokenizer=tokenizer, data=ds_train)
ds_test = InstructionDataset(tokenizer=tokenizer, data=ds_test)
ds_val = InstructionDataset(tokenizer=tokenizer, data=ds_val)

In [6]:

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)  # add batch dimension
    return encoded_tensor


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)  # remove batch dimension
    return tokenizer.decode(flat.tolist())


In [7]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]


        padded = (                               #1
            new_item + [pad_token_id] *          #1
            (batch_max_length - len(new_item))   #1
        )
        inputs = torch.tensor(padded[:-1])      #2
        targets = torch.tensor(padded[1:])     #3

        mask = targets == pad_token_id              #4
        indices = torch.nonzero(mask).squeeze()     #4
        if indices.numel() > 1:                     #4
            targets[indices[1:]] = ignore_index     #4

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]       #5
            targets = targets[:allowed_max_length]     #5

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [8]:
from torch.utils.data import DataLoader

num_workers = 0      #1
batch_size = 8


train_loader = DataLoader(
    ds_train,
    batch_size=batch_size,
    collate_fn=custom_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

val_loader = DataLoader(
    ds_val,
    batch_size=batch_size,
    collate_fn=custom_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_loader = DataLoader(
    ds_test,
    batch_size=batch_size,
    collate_fn=custom_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [9]:
print("Train loader:")
for inputs, targets in val_loader:
    print(inputs.shape, targets.shape)

Train loader:
torch.Size([8, 334]) torch.Size([8, 334])
torch.Size([8, 470]) torch.Size([8, 470])
torch.Size([8, 564]) torch.Size([8, 564])
torch.Size([8, 373]) torch.Size([8, 373])
torch.Size([8, 307]) torch.Size([8, 307])
torch.Size([8, 275]) torch.Size([8, 275])
torch.Size([8, 521]) torch.Size([8, 521])
torch.Size([8, 192]) torch.Size([8, 192])
torch.Size([8, 398]) torch.Size([8, 398])
torch.Size([8, 500]) torch.Size([8, 500])
torch.Size([8, 433]) torch.Size([8, 433])
torch.Size([8, 517]) torch.Size([8, 517])
torch.Size([8, 351]) torch.Size([8, 351])
torch.Size([8, 402]) torch.Size([8, 402])
torch.Size([8, 253]) torch.Size([8, 253])
torch.Size([8, 400]) torch.Size([8, 400])
torch.Size([8, 392]) torch.Size([8, 392])
torch.Size([8, 430]) torch.Size([8, 430])
torch.Size([8, 349]) torch.Size([8, 349])
torch.Size([8, 442]) torch.Size([8, 442])
torch.Size([8, 335]) torch.Size([8, 335])
torch.Size([8, 441]) torch.Size([8, 441])
torch.Size([8, 497]) torch.Size([8, 497])
torch.Size([8, 421])

In [10]:
from tgedr_lm.commons.utils_io import load_pickle_compressed

weights = load_pickle_compressed(PATH_GPT2_124M_WEIGHTS)

In [11]:
"""GPT model definition and supporting layers for language modeling."""

from abc import abstractmethod
import logging
import torch
from torch import nn
import numpy as np

from transformers import PreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput
from tgedr_lm.configuration import ClassifierBaseConfiguration
from tgedr_lm.layers.blocks import TransformerBlock
from tgedr_lm.layers.normalization import LayerNormalization

class GPT2INstructionBase(PreTrainedModel):
    """Base class for GPT-2 instruction models.

    This abstract base class provides common functionality for GPT-2 based
    instruction models, including model initialization, device management, metrics
    computation, and loss calculation.
    """

    def __init__(self, cfg: ClassifierBaseConfiguration) -> None:
        """Initialize GPT-2 model layers from the given configuration.

        Parameters
        ----------
        cfg : ClassifierBaseConfiguration
            Model configuration containing vocabulary size, context length,
            embedding dimension, dropout rate, and number of transformer layers.
        """
        super().__init__(cfg)
        self.tok_emb = nn.Embedding(cfg.vocabulary_size, cfg.embeddings_dimension)
        self.pos_emb = nn.Embedding(cfg.context_length, cfg.embeddings_dimension)

        self.drop_emb = nn.Dropout(cfg.drop_rate)

        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg.n_layers)])

        self.final_norm = LayerNormalization(cfg.embeddings_dimension)
        self.out_head = nn.Linear(cfg.embeddings_dimension, cfg.vocabulary_size, bias=False)


    @staticmethod
    def compute_metrics(eval_pred) -> dict[str, float]:
        """Compute accuracy, macro precision, macro recall, and macro F-score from logits."""
        logits, labels = eval_pred

        if isinstance(logits, tuple):
            logits = logits[0]

        if isinstance(logits, torch.Tensor):
            logits = logits.detach().cpu().numpy()

        if isinstance(labels, torch.Tensor):
            labels = labels.detach().cpu().numpy()

        if labels.ndim > 1:
            labels = np.argmax(labels, axis=-1)

        labels = labels.reshape(-1)

        if logits.ndim == 3:
            logits = logits[:, -1, :]

        preds = np.argmax(logits, axis=-1).reshape(-1)

        classes = np.unique(np.concatenate((labels, preds)))
        precisions = []
        recalls = []
        f_scores = []

        for cls in classes:
            tp = np.sum((preds == cls) & (labels == cls))
            fp = np.sum((preds == cls) & (labels != cls))
            fn = np.sum((preds != cls) & (labels == cls))

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

            precisions.append(float(precision))
            recalls.append(float(recall))
            f_scores.append(float(f_score))

        macro_precision = float(np.mean(precisions)) if precisions else 0.0
        macro_recall = float(np.mean(recalls)) if recalls else 0.0
        macro_f = float(np.mean(f_scores)) if f_scores else 0.0

        return {
            "accuracy": float((preds == labels).mean()),
            "precision": macro_precision,
            "recall": macro_recall,
            "f": macro_f,
        }

    def compute_logits(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Compute classifier logits for a batch of token ids."""
        _, seq_len = input_ids.shape
        input_ids = input_ids.to(self.device)
        tok_embeds = self.tok_emb(input_ids)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=self.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)

    def calculate_batch_loss(
        self,
        input_batch: torch.Tensor,
        target_batch: torch.Tensor,
        device: torch.device | None = None,
    ) -> torch.Tensor:
        """Calculate cross-entropy loss for a batch of inputs and targets.

        Parameters
        ----------
        input_batch : torch.Tensor
            The input tensor batch.
        target_batch : torch.Tensor
            The target tensor batch.
        device : torch.device | None, optional
            The device to compute on. If None, uses self.device.

        Returns
        -------
        torch.Tensor
            The cross-entropy loss for the batch.
        """
        logger.debug(f"[calculate_batch_loss|in] ({input_batch}, {target_batch})")
        target_device = self.device if device is None else device
        input_batch = input_batch.to(target_device)
        target_batch = target_batch.to(target_device)
        logits = self.compute_logits(input_batch)[:, -1, :]
        loss = torch.nn.functional.cross_entropy(logits, target_batch)
        logger.debug(f"[calculate_batch_loss|out] => {loss}")
        return loss

    def infer(self, in_idx: torch.Tensor) -> torch.Tensor:
        """Perform a forward pass through the model to obtain class logits.

        Parameters
        ----------
        in_idx : torch.Tensor
            Input tensor of shape (batch_size, sequence_length) containing token indices.

        Returns
        -------
        torch.Tensor
            Predicted class indices.
        """
        in_idx = in_idx.to(self.device)
        was_training = self.training
        if was_training:
            self.eval()
        with torch.no_grad():  # Models inference without gradient tracking
            logits = self.compute_logits(in_idx)[:, -1, :]
        if was_training:
            self.train()
        return torch.argmax(logits, dim=-1)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
        **kwargs: any,
    ) -> SequenceClassifierOutput:
        """Forward pass compatible with Hugging Face Trainer.

        Parameters
        ----------
        input_ids : torch.Tensor
            Input token indices of shape (batch_size, sequence_length).
        attention_mask : torch.Tensor, optional
            Attention mask (not currently used by GPT2Classifier).
        labels : torch.Tensor, optional
            Target class labels for computing loss.
        **kwargs
            Additional arguments (ignored for compatibility).

        Returns
        -------
        SequenceClassifierOutput
            Hugging Face classifier output containing logits and optionally loss.
        """
        del attention_mask, kwargs
        logits = self.compute_logits(input_ids)
        loss = None

        if labels is not None:
            labels = labels.to(self.device)
            loss = torch.nn.functional.cross_entropy(logits[:, -1, :], labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)

    def pretrain(self, weights: dict) -> None:
        """Prepare the model for fine-tuning by loading pretrained weights.

        Parameters
        ----------
        weights : dict
            Dictionary containing pretrained weights to load into the model.
        """
        self._load_weights(weights)
        for param in self.parameters():
            param.requires_grad = False
        # fine-tuning additional layers can noticeably improve the predictive performance of the model
        # We also configure the last transformer block and the final LayerNorm module,
        # which connects this block to the output layer, to be trainable
        for param in self.trf_blocks[-1].parameters():
            param.requires_grad = True
        for param in self.final_norm.parameters():
            param.requires_grad = True
        for param in self.out_head.parameters():
            param.requires_grad = True

    def _assign(self, left, right) -> torch.nn.Parameter:
        """Validate shapes and return right as a new Parameter with the same shape as left.

        Parameters
        ----------
        left : torch.Tensor
            Reference tensor whose shape must match right.
        right : array-like
            Source data to wrap as a parameter.

        Returns
        -------
        torch.nn.Parameter
            Parameter wrapping the values of right.

        Raises
        ------
        ValueError
            If left and right have different shapes.
        """
        if left.shape != right.shape:
            raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")  # noqa: EM102, TRY003
        return torch.nn.Parameter(torch.tensor(right))

    def _load_weights(self, params: dict) -> None:
        """Load pretrained weights into the model from a parameters dictionary.

        Parameters
        ----------
        params : dict
            Dictionary containing pretrained weights for token embeddings, positional
            embeddings, transformer blocks, and output layer.
        """
        # 1 Sets the model's positional and token embedding weights to those specified in params.

        self.pos_emb.weight = self._assign(self.pos_emb.weight, params["wpe"])
        self.tok_emb.weight = self._assign(self.tok_emb.weight, params["wte"])

        for b in range(len(params["blocks"])):  # 2 Iterates over each transformer block in the model
            # 3 The np.split function is used to divide the attention and bias weights
            # into three equal parts for the query, key, and value components.
            q_w, k_w, v_w = np.split((params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
            self.trf_blocks[b].att.W_query.weight = self._assign(self.trf_blocks[b].att.W_query.weight, q_w.T)
            self.trf_blocks[b].att.W_key.weight = self._assign(self.trf_blocks[b].att.W_key.weight, k_w.T)
            self.trf_blocks[b].att.W_value.weight = self._assign(self.trf_blocks[b].att.W_value.weight, v_w.T)

            q_b, k_b, v_b = np.split((params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
            self.trf_blocks[b].att.W_query.bias = self._assign(self.trf_blocks[b].att.W_query.bias, q_b)
            self.trf_blocks[b].att.W_key.bias = self._assign(self.trf_blocks[b].att.W_key.bias, k_b)
            self.trf_blocks[b].att.W_value.bias = self._assign(self.trf_blocks[b].att.W_value.bias, v_b)

            self.trf_blocks[b].att.out_projection.weight = self._assign(
                self.trf_blocks[b].att.out_projection.weight, params["blocks"][b]["attn"]["c_proj"]["w"].T
            )
            self.trf_blocks[b].att.out_projection.bias = self._assign(
                self.trf_blocks[b].att.out_projection.bias, params["blocks"][b]["attn"]["c_proj"]["b"]
            )

            self.trf_blocks[b].ff.layers[0].weight = self._assign(
                self.trf_blocks[b].ff.layers[0].weight, params["blocks"][b]["mlp"]["c_fc"]["w"].T
            )
            self.trf_blocks[b].ff.layers[0].bias = self._assign(
                self.trf_blocks[b].ff.layers[0].bias, params["blocks"][b]["mlp"]["c_fc"]["b"]
            )
            self.trf_blocks[b].ff.layers[2].weight = self._assign(
                self.trf_blocks[b].ff.layers[2].weight, params["blocks"][b]["mlp"]["c_proj"]["w"].T
            )
            self.trf_blocks[b].ff.layers[2].bias = self._assign(
                self.trf_blocks[b].ff.layers[2].bias, params["blocks"][b]["mlp"]["c_proj"]["b"]
            )

            self.trf_blocks[b].norm1.scale = self._assign(
                self.trf_blocks[b].norm1.scale, params["blocks"][b]["ln_1"]["g"]
            )
            self.trf_blocks[b].norm1.shift = self._assign(
                self.trf_blocks[b].norm1.shift, params["blocks"][b]["ln_1"]["b"]
            )
            self.trf_blocks[b].norm2.scale = self._assign(
                self.trf_blocks[b].norm2.scale, params["blocks"][b]["ln_2"]["g"]
            )
            self.trf_blocks[b].norm2.shift = self._assign(
                self.trf_blocks[b].norm2.shift, params["blocks"][b]["ln_2"]["b"]
            )

        self.final_norm.scale = self._assign(self.final_norm.scale, params["g"])
        self.final_norm.shift = self._assign(self.final_norm.shift, params["b"])


In [12]:
class GPTModel(nn.Module):
    def __init__(self, cfg: ClassifierBaseConfiguration):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg.vocabulary_size, cfg.embeddings_dimension)
        self.pos_emb = nn.Embedding(cfg.context_length, cfg.embeddings_dimension)

        self.drop_emb = nn.Dropout(cfg.drop_rate)

        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg.n_layers)])

        self.final_norm = LayerNormalization(cfg.embeddings_dimension)
        self.out_head = nn.Linear(cfg.embeddings_dimension, cfg.vocabulary_size, bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits
    
    def pretrain(self, weights: dict) -> None:
        """Prepare the model for fine-tuning by loading pretrained weights.

        Parameters
        ----------
        weights : dict
            Dictionary containing pretrained weights to load into the model.
        """
        self._load_weights(weights)
        for param in self.parameters():
            param.requires_grad = False
        # fine-tuning additional layers can noticeably improve the predictive performance of the model
        # We also configure the last transformer block and the final LayerNorm module,
        # which connects this block to the output layer, to be trainable
        for param in self.trf_blocks[-1].parameters():
            param.requires_grad = True
        for param in self.final_norm.parameters():
            param.requires_grad = True
        for param in self.out_head.parameters():
            param.requires_grad = True

    def _assign(self, left, right) -> torch.nn.Parameter:
        """Validate shapes and return right as a new Parameter with the same shape as left.

        Parameters
        ----------
        left : torch.Tensor
            Reference tensor whose shape must match right.
        right : array-like
            Source data to wrap as a parameter.

        Returns
        -------
        torch.nn.Parameter
            Parameter wrapping the values of right.

        Raises
        ------
        ValueError
            If left and right have different shapes.
        """
        if left.shape != right.shape:
            raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")  # noqa: EM102, TRY003
        return torch.nn.Parameter(torch.tensor(right))

    def _load_weights(self, params: dict) -> None:
        """Load pretrained weights into the model from a parameters dictionary.

        Parameters
        ----------
        params : dict
            Dictionary containing pretrained weights for token embeddings, positional
            embeddings, transformer blocks, and output layer.
        """
        # 1 Sets the model's positional and token embedding weights to those specified in params.

        self.pos_emb.weight = self._assign(self.pos_emb.weight, params["wpe"])
        self.tok_emb.weight = self._assign(self.tok_emb.weight, params["wte"])

        for b in range(len(params["blocks"])):  # 2 Iterates over each transformer block in the model
            # 3 The np.split function is used to divide the attention and bias weights
            # into three equal parts for the query, key, and value components.
            q_w, k_w, v_w = np.split((params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
            self.trf_blocks[b].att.W_query.weight = self._assign(self.trf_blocks[b].att.W_query.weight, q_w.T)
            self.trf_blocks[b].att.W_key.weight = self._assign(self.trf_blocks[b].att.W_key.weight, k_w.T)
            self.trf_blocks[b].att.W_value.weight = self._assign(self.trf_blocks[b].att.W_value.weight, v_w.T)

            q_b, k_b, v_b = np.split((params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
            self.trf_blocks[b].att.W_query.bias = self._assign(self.trf_blocks[b].att.W_query.bias, q_b)
            self.trf_blocks[b].att.W_key.bias = self._assign(self.trf_blocks[b].att.W_key.bias, k_b)
            self.trf_blocks[b].att.W_value.bias = self._assign(self.trf_blocks[b].att.W_value.bias, v_b)

            self.trf_blocks[b].att.out_projection.weight = self._assign(
                self.trf_blocks[b].att.out_projection.weight, params["blocks"][b]["attn"]["c_proj"]["w"].T
            )
            self.trf_blocks[b].att.out_projection.bias = self._assign(
                self.trf_blocks[b].att.out_projection.bias, params["blocks"][b]["attn"]["c_proj"]["b"]
            )

            self.trf_blocks[b].ff.layers[0].weight = self._assign(
                self.trf_blocks[b].ff.layers[0].weight, params["blocks"][b]["mlp"]["c_fc"]["w"].T
            )
            self.trf_blocks[b].ff.layers[0].bias = self._assign(
                self.trf_blocks[b].ff.layers[0].bias, params["blocks"][b]["mlp"]["c_fc"]["b"]
            )
            self.trf_blocks[b].ff.layers[2].weight = self._assign(
                self.trf_blocks[b].ff.layers[2].weight, params["blocks"][b]["mlp"]["c_proj"]["w"].T
            )
            self.trf_blocks[b].ff.layers[2].bias = self._assign(
                self.trf_blocks[b].ff.layers[2].bias, params["blocks"][b]["mlp"]["c_proj"]["b"]
            )

            self.trf_blocks[b].norm1.scale = self._assign(
                self.trf_blocks[b].norm1.scale, params["blocks"][b]["ln_1"]["g"]
            )
            self.trf_blocks[b].norm1.shift = self._assign(
                self.trf_blocks[b].norm1.shift, params["blocks"][b]["ln_1"]["b"]
            )
            self.trf_blocks[b].norm2.scale = self._assign(
                self.trf_blocks[b].norm2.scale, params["blocks"][b]["ln_2"]["g"]
            )
            self.trf_blocks[b].norm2.shift = self._assign(
                self.trf_blocks[b].norm2.shift, params["blocks"][b]["ln_2"]["b"]
            )

        self.final_norm.scale = self._assign(self.final_norm.scale, params["g"])
        self.final_norm.shift = self._assign(self.final_norm.shift, params["b"])


In [13]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):

    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # New: Filter logits with top_k sampling
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # New: Apply temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # New (not in book): numerical stability tip to get equivalent results on mps device
            # subtract rowwise max before softmax
            logits = logits - logits.max(dim=-1, keepdim=True).values

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:  # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

In [14]:
cfg = ClassifierBaseConfiguration(n_classes=None)
model = GPTModel(cfg)
model.pretrain(weights)


In [15]:
input_text = "### Instruction:\nConvert the active sentence to passive: 'The chef cooks the meal every day.'"

In [16]:
text_to_token_ids(input_text, tokenizer)

tensor([[21017, 46486,    25,   198,  3103,  1851,   262,  4075,  6827,   284,
         14513,    25,   705,   464, 21221, 38383,   262,  9799,   790,  1110,
          2637]])

In [17]:
token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=cfg.context_length,
    eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)

In [18]:
generated_text


"### Instruction:\nConvert the active sentence to passive: 'The chef cooks the meal every day.' annex Monsters Knicks Knicks Knicks Knicks Knicks Knicks Knicksaila annex Monstersaila Tip annex Monsters hurricane annex Monsters rewarding annex Monsters Knicks Knicksichever annex annex annex Monsters annexailaaid annexailaaila"

In [19]:
response_text = generated_text[len(input_text):].strip()
print(response_text)

annex Monsters Knicks Knicks Knicks Knicks Knicks Knicks Knicksaila annex Monstersaila Tip annex Monsters hurricane annex Monsters rewarding annex Monsters Knicks Knicksichever annex annex annex Monsters annexailaaid annexailaaila


In [20]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with torch.no_grad():
    train_loss = calc_loss_loader(
        train_loader, model, device, num_batches=5
    )
    val_loss = calc_loss_loader(
        val_loader, model, device, num_batches=5
)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 18.44499282836914
Validation loss: 18.522509002685545


In [ ]:

def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # Focus only on the last time step
        # (batch, n_token, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Get the idx of the vocab entry with the highest logits value
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()

def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode
        
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            tokens_seen += input_batch.numel()
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Print a sample text after each epoch
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen


In [ ]:
import time

start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=0.00005, weight_decay=0.1
)
num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")